# Hash Tables

## 1. Hash Table Fundamentals

A hash table (or hash map) stores key-value pairs, using a hash function to compute an index into an array of slots (buckets).

- **Key-value pairs:** each unique key maps to a value (e.g. `"alice" -> {name: "Alice", email: "alice@example.com"}`)
- **Hash function:** converts a key into a fixed-size integer (hash code), deterministic for the same input
- **Index mapping:** the hash code is mapped to a slot index, typically via `hash(key) % table_size`
- **Collisions:** occur when two keys produce the same index, resolved via chaining or open addressing

Without a hash table, lookup requires scanning entries one by one -- O(n). A hash table computes the index directly from the key to give O(1) average access.

### Terminology

| Term | Definition |
|------|-----------|
| Slot (bucket) | A position in the underlying array that can hold one or more entries |
| Hash function | Maps a key to a slot index: `Hash(key) -> index in [0, m-1]` |
| Hash collision | Two different keys map to the same index |
| Load factor | $\alpha = n / m$ where n = stored items, m = table size |
| Rehash | Resize the table and recompute indices for all entries when load factor exceeds threshold |
| Probing | The process of finding an available bucket when a collision occurs (open addressing) |

### Hash Function Properties

A good hash function must be:

- **Deterministic:** same key always produces the same hash code
- **Uniform:** produces a wide, evenly distributed range of outputs to minimize collisions
- **Fast:** computable in O(1) for fixed-size keys -- every table operation hashes the key first

### Properties

| Property | Detail |
|---|---|
| **Key idea** | Constant-time key-value access via computed index |
| **Use-case** | Frequency counting, dedup, caching, symbol tables |
| **Time** | O(1) avg insert/search/delete, O(n) worst |
| **Space** | O(n) |

### Operations

| Operation | Description | Time (avg) |
|-----------|-------------|-----------|
| `put(key, val)` | Insert or update key-value pair | O(1) |
| `get(key)` | Return value for key, or None | O(1) |
| `remove(key)` | Delete key-value pair | O(1) |
| `__len__()` | Number of stored pairs | O(1) |
| `__contains__(key)` | Check if key exists | O(1) |

### Hash Table (Chaining)

In [1]:
class HashTable:
    def __init__(self, capacity=13):
        self.capacity = capacity
        self.size = 0
        self.table = [[] for _ in range(capacity)]

    def _hash(self, key):
        return key % self.capacity

    def put(self, key, val):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx][i] = (key, val)
                return
        self.table[idx].append((key, val))
        self.size += 1

    def get(self, key):
        idx = self._hash(key)
        for k, v in self.table[idx]:
            if k == key:
                return v
        return None

    def remove(self, key):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx].pop(i)
                self.size -= 1
                return

    def __len__(self):
        return self.size

    def __contains__(self, key):
        return self.get(key) is not None

#### Steps

`Hash(key) = key % 13`, chaining on collision.

| Step | Operation | Index | Chain at index | Notes |
|------|-----------|-------|---------------|-------|
| 1 | `H.put(88, "a")` | `88 % 13 = 10` | `[(88, "a")]` | -- |
| 2 | `H.put(60, "b")` | `60 % 13 = 8` | `[(60, "b")]` | -- |
| 3 | `H.put(65, "c")` | `65 % 13 = 0` | `[(65, "c")]` | -- |
| 4 | `H.put(39, "d")` | `39 % 13 = 0` | `[(65, "c"), (39, "d")]` | collision with 65, chained |
| 5 | `H.get(65)` | `65 % 13 = 0` | scan chain | found at chain[0] -> `"c"` |
| 6 | `H.get(99)` | `99 % 13 = 8` | scan chain | no key 99 -> `None` |
| 7 | `H.remove(65)` | `65 % 13 = 0` | `[(39, "d")]` | removed from chain |
| 8 | `65 in H` | `65 % 13 = 0` | scan chain | not found -> `False` |

### Python dict as Hash Table

Python's `dict` is a hash table using open addressing internally.

| ADT Operation | Python dict | Time |
|--------------|-------------|------|
| `put(key, val)` | `d[key] = val` | O(1) avg |
| `get(key)` | `d.get(key)` or `d[key]` | O(1) avg |
| `remove(key)` | `del d[key]` | O(1) avg |
| `__len__()` | `len(d)` | O(1) |
| `__contains__(key)` | `key in d` | O(1) avg |

## 2. Hash Functions

A hash function takes a key and converts it to a fixed-size integer, then maps it to a table index.

### Common Methods

| Method | Formula | Notes |
|--------|---------|-------|
| Direct addressing | $H(k) = k$ or $H(k) = ak + b$ | One-to-one mapping, requires space proportional to key range |
| Division (modulo) | $H(k) = k \bmod m$ | Choose $m$ as prime to reduce clustering from key patterns |
| Mid-square | Square key, extract middle digits | Mixes all bits of the key, producing uniform spread |
| Folding | Split key into equal parts, sum | Combines all segments of long keys (e.g. phone numbers) |
| Polynomial | $H(s) = \sum_{i=0}^{n-1} s[i] \cdot d^{n-1-i} \bmod q$ | Position-sensitive string hashing via Horner's method |

### Division Method

Most commonly used. Compute $H(k) = k \bmod m$ where $m$ is a prime number.

- Avoid $m$ as a power of 2 -- only the lowest-order bits affect the hash
- Example: $H(88) = 88 \bmod 13 = 10$

### Polynomial Hash (Strings)

For string keys, treat each character as a coefficient in a polynomial.

$$H(s) = \left(\sum_{i=0}^{n-1} s[i] \cdot d^{n-1-i}\right) \bmod q$$

- Horner's method computes iteratively: $H = ((H \cdot d) + \text{ord}(c)) \bmod q$
- $d$ is a prime base (e.g. 31), $q$ is a large prime modulus
- Position-dependent: `"abc"` and `"cba"` produce different hashes
- Used in Rabin-Karp substring search (see `strings.ipynb`)

## 3. Collision Resolution

A collision occurs when two different keys hash to the same index.

- Collisions are inevitable when the number of possible keys exceeds the table size (pigeonhole principle)
- Two main strategies:
  - **Open addressing:** all entries live in the bucket array, one per bucket. On collision, probe for the next empty bucket within the array.
  - **Chaining:** each bucket holds a linked list of entries. On collision, append to the list at that bucket.

### Open Addressing

All entries stored in the bucket array, one per bucket. On collision, probe for the next empty slot.

| Probing | Formula | Behavior |
|---------|---------|----------|
| Linear | `H(i) = (Hash(key) + i) % m` | Steps sequentially, prone to clustering |
| Quadratic | `H(i) = (Hash(key) + c1*i + c2*i^2) % m` | Larger jumps reduce clustering |
| Double hashing | `H(i) = (Hash(key) + i * Hash2(key)) % m` | Second hash determines step size |

- **Clustering:** collisions fill adjacent slots into a block. New keys hashing into the block must probe past all of it, growing the block further and slowing subsequent operations.
- **Prime table size:** non-prime `m` can cause probes to cycle through only a subset of slots (e.g. `m=8` with step 2 only visits even indices). Prime `m` guarantees all slots are reachable.
- **Deletion:** emptying a slot would cause probes that previously passed through it to stop early, missing entries beyond. Instead, mark as "deleted" so probes continue past but insertions can reuse.
- **Load factor:** as $\alpha$ approaches 1.0, fewer empty slots remain, so probe sequences grow longer and operations approach O(n). Typically resize when $\alpha > 0.75$.

#### Steps

Linear probing: on collision, check the next slot sequentially (`i+1, i+2, ...`), wrapping around to the start of the array if needed.

`Hash(key) = key % 11`, table size m = 11. Insert keys 28, 49, 18, 38.

| Step | Key | Hash | Probe Sequence | Stored At | Table State |
|------|-----|------|---------------|-----------|-------------|
| 1 | 28 | `28 % 11 = 6` | 6 (empty) | 6 | `[_, _, _, _, _, _, 28, _, _, _, _]` |
| 2 | 49 | `49 % 11 = 5` | 5 (empty) | 5 | `[_, _, _, _, _, 49, 28, _, _, _, _]` |
| 3 | 18 | `18 % 11 = 7` | 7 (empty) | 7 | `[_, _, _, _, _, 49, 28, 18, _, _, _]` |
| 4 | 38 | `38 % 11 = 5` | 5 (full), 6 (full), 7 (full), 8 (empty) | 8 | `[_, _, _, _, _, 49, 28, 18, 38, _, _]` |

Key 38 hashes to slot 5, probes forward through occupied slots 5, 6, 7 to reach empty slot 8. Slots 5-8 now form a cluster.

### Chaining

Each slot holds a list (chain) of all key-value pairs that hash to the same index.

- Insert: hash key to index, append pair to the chain
- Search: hash key to index, scan the chain for matching key
- Delete: hash key to index, remove matching pair from the chain

### Comparison

| Method | Insert | Search (avg) | Delete | Space |
|--------|--------|-------------|--------|-------|
| Open addressing | O(1) avg | $O(1/(1-\alpha))$ | $O(1/(1-\alpha))$ | O(m) |
| Chaining | O(1) | $O(1 + \alpha)$ | $O(1 + \alpha)$ | O(n + m) |

Where $\alpha = n/m$ is the load factor. Chaining degrades more predictably as $\alpha$ increases.

## 4. Dynamic Resizing

A dynamic hash table resizes when the load factor crosses a threshold.

- Grow when $\alpha > 0.75$ -- double the capacity
- Shrink when $\alpha < 0.25$ -- halve the capacity
- On resize, allocate a new table and rehash all entries
- Rehash costs O(n) per call but occurs infrequently -- amortized O(1) per insert

In [ ]:
class DynamicHashTable:
    def __init__(self, capacity=13):
        self.capacity = capacity
        self.size = 0
        self.table = [[] for _ in range(capacity)]

    def _hash(self, key):
        return key % self.capacity

    def _resize(self, newCapacity):
        oldTable = self.table
        self.capacity = newCapacity
        self.size = 0
        self.table = [[] for _ in range(newCapacity)]
        for chain in oldTable:
            for key, val in chain:
                self.put(key, val)

    def put(self, key, val):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx][i] = (key, val)
                return
        self.table[idx].append((key, val))
        self.size += 1
        if self.size / self.capacity > 0.75:
            self._resize(self.capacity * 2)

    def get(self, key):
        idx = self._hash(key)
        for k, v in self.table[idx]:
            if k == key:
                return v
        return None

    def remove(self, key):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx].pop(i)
                self.size -= 1
                return

#### Steps

`DynamicHashTable(capacity=4)`, insert keys until resize triggers.

| Step | Operation | size/capacity | $\alpha$ | Action |
|------|-----------|--------------|----------|--------|
| 1 | `put(5, "a")` | 1/4 | 0.25 | insert at slot 1 |
| 2 | `put(10, "b")` | 2/4 | 0.50 | insert at slot 2 |
| 3 | `put(3, "c")` | 3/4 | 0.75 | insert at slot 3 |
| 4 | `put(7, "d")` | 4/4 | 1.00 | insert, $\alpha > 0.75$ -> resize to capacity 8 |
| -- | `_resize(8)` | 4/8 | 0.50 | rehash all 4 entries into new table |
| 5 | `put(15, "e")` | 5/8 | 0.625 | insert into resized table |

## 5. Applications

Common hash table patterns in algorithm problems:

| Pattern | Description | Problem keywords |
|---------|-------------|-----------------|
| **Frequency counting** | Count occurrences of each element | "most frequent", "top k", "majority element" |
| **Complement lookup** | Check if a target complement exists in O(1) | "two sum", "pair with sum", "subarray sum" |
| **Deduplication** | Track seen elements to filter duplicates | "contains duplicate", "first unique", "distinct" |
| **Grouping** | Group items by a computed key | "group anagrams", "group by", "categorize" |

### Frequency Counting

In [ ]:
def countFrequency(arr):
    freq = {}
    for val in arr:
        freq[val] = freq.get(val, 0) + 1
    return freq


print(countFrequency([1, 2, 2, 3, 3, 3]))

### Two-Sum Pattern

In [ ]:
def twoSum(nums, target):
    hashmap = {}
    for i, val in enumerate(nums):
        complement = target - val
        if complement in hashmap:
            return [hashmap[complement], i]
        hashmap[val] = i
    return []


print(twoSum([2, 7, 11, 15], 9))

#### Steps

`twoSum([2, 7, 11, 15], target=9)`

| Step | val | complement | hashmap | Result |
|------|-----|-----------|---------|--------|
| 1 | 2 | 7 | `{2: 0}` | 7 not in hashmap |
| 2 | 7 | 2 | `{2: 0, 7: 1}` | 2 in hashmap -> return `[0, 1]` |

## 6. Reference

### Hash Function Comparison

| Method | Time | Collisions | Best for |
|--------|------|-----------|----------|
| Direct addressing | O(1) | None | Small, dense key ranges |
| Division (modulo) | O(1) | Moderate (prime m helps) | Integer keys |
| Polynomial | O(L) where L = key length | Low with good d, q | String keys |

### Collision Resolution Comparison

| Method | Insert | Search (avg) | Delete | Space | Notes |
|--------|--------|-------------|--------|-------|-------|
| Linear probing | O(1) | $O(1/(1-\alpha))$ | Requires deleted flag | O(m) | Prone to clustering |
| Quadratic probing | O(1) | $O(1/(1-\alpha))$ | Requires deleted flag | O(m) | Reduces clustering |
| Chaining | O(1) | $O(1 + \alpha)$ | O(1) to unlink | O(n + m) | Predictable degradation |

### Hash Table vs Other Structures

| Structure | Insert | Search | Delete | Ordered | Space |
|-----------|--------|--------|--------|---------|-------|
| Hash table | O(1) avg | O(1) avg | O(1) avg | No | O(n) |
| BST | O(log n) | O(log n) | O(log n) | Yes | O(n) |
| Sorted array | O(n) | O(log n) | O(n) | Yes | O(n) |
| Unsorted array | O(1) | O(n) | O(n) | No | O(n) |

### Cross-References

- Rabin-Karp substring search (polynomial rolling hash): see `strings.ipynb`
- Hash set / hash map usage in LeetCode: see `pset/leetcode/arrays/`